<a href="https://colab.research.google.com/github/hobbsandbobs-dotcom/Analytics-Coursework/blob/main/1_Data_importing_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INITIAL ENVIRONMENT SET UP

1.1 Dataset uploading and library imports

In [ ]:
import pandas as pd

#Uploading NorthStar's datasets into Colab environment
from google.colab import files
uploaded_files = files.upload()

Saving nstar_app_events.csv to nstar_app_events.csv
Saving nstar_complaints.csv to nstar_complaints.csv
Saving nstar_customers.csv to nstar_customers.csv
Saving nstar_data_dictionary.csv to nstar_data_dictionary.csv
Saving nstar_deliveries.csv to nstar_deliveries.csv
Saving nstar_drivers.csv to nstar_drivers.csv
Saving nstar_hubs.csv to nstar_hubs.csv
Saving nstar_incidents.csv to nstar_incidents.csv
Saving nstar_orders.csv to nstar_orders.csv
Saving nstar_vehicles.csv to nstar_vehicles.csv


1.2. Creating functions

In [ ]:
#Function for cleaning zones using a panda series and if else
def zone_cleaning(zone_column):

    if hasattr(zone_column, "str"):

        return (
            zone_column.astype(str)
               .str.strip()
               .str.title()
               .replace({
                   'Ctr': 'Central'
               })
        )

    else:
        zone_column = str(zone_column).strip().replace("_", " ").title()
        zone_replacements = {
            'Ctr': 'Central'
        }

        return zone_replacements.get(zone_column, zone_column)

In [ ]:
#Function for standardising the table format
def format_table(report_table, title, cmap="Purples"):
    report_table = report_table.copy()

    report_table.index = [zone_cleaning(x) for x in report_table.index]
    report_table.columns = [zone_cleaning(x) for x in report_table.columns]

    return (
        report_table.style
        .set_caption(title)
        .format(precision=2)
        .background_gradient(cmap=cmap)
        .set_properties(**{
            "text-align": "center",
            "font-size": "11pt"
        })
        .set_table_styles([
            {'selector': 'caption',
             'props': [
                 ('font-size', '12pt'),
                 ('font-weight', 'bold'),
                 ('text-align', 'center')
             ]},
            {'selector': 'th.col_heading',
             'props': [
                 ('font-size', '10pt'),
                 ('font-weight', 'normal')
             ]}
        ])
    )


In [ ]:
#Function for standardising the label format
def clean_label(label_name):
    return str(label_name).strip().replace("_", " ").title()

# **2. Cleaning the data**

2.1. Customers dataset

In [ ]:
customers_df = pd.read_csv("nstar_customers.csv")

customers_df.head()

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61,AIRPORT,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75,CENTRAL,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


In [ ]:
customers_df.info()

"Duplicates?", customers_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           650 non-null    object 
 1   age                   650 non-null    int64  
 2   home_zone             650 non-null    object 
 3   customer_type         650 non-null    object 
 4   signup_date           650 non-null    object 
 5   loyalty_score         630 non-null    float64
 6   app_engagement_score  650 non-null    float64
 7   preferred_channel     637 non-null    object 
 8   account_status        650 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 45.8+ KB


('Duplicates?', np.int64(0))

In [ ]:
missing_before = customers_df.isnull().sum().to_frame(name="Missing N.")

format_table(missing_before, "Missing Values: Customers")

,Missing N.
Customer Id,0
Age,0
Home Zone,0
Customer Type,0
Signup Date,0
Loyalty Score,20
App Engagement Score,0
Preferred Channel,13
Account Status,0


In [ ]:
customers_df["loyalty_score"].describe()

,loyalty_score
count,650.000000
mean,59.690635
std,15.874276
min,13.100000
25%,49.300000
50%,59.690635
75%,69.875000
max,99.000000


In [ ]:
customers_df["preferred_channel"].describe()

,preferred_channel
count,637
unique,4
top,App
freq,367


In [ ]:
#As the spread was moderate, had an acceptable range, and a close mean and median, .mean() was chosen to keep loyalty_score numeric integrity
customers_df["loyalty_score"] = customers_df["loyalty_score"].fillna(
    customers_df["loyalty_score"].mean()
)

#As preferred_channel is categorical, "Unknown" was used to fill the null values
customers_df["preferred_channel"] = customers_df["preferred_channel"].fillna("Unknown")

In [ ]:
missing_after = customers_df.isnull().sum().to_frame(name="Missing Values")

format_table(missing_after, "Missing Values: Post Cleaning")

,Missing Values
Customer Id,0
Age,0
Home Zone,0
Customer Type,0
Signup Date,0
Loyalty Score,0
App Engagement Score,0
Preferred Channel,0
Account Status,0


In [ ]:
customers_df["home_zone"].value_counts()

,count
home_zone,
SOUTH,50
RiverSide,49
East,48
WEST,45
CENTRAL,44
West,43
South,42
Riverside,42
EAST,41


In [ ]:
customers_df["home_zone"] = zone_cleaning(customers_df["home_zone"])

In [ ]:
customers_df["home_zone"].value_counts()

,count
home_zone,
North,111
Central,110
South,92
Riverside,91
East,89
West,88
Airport,69


2.2. Orders dataset

In [ ]:
orders_df = pd.read_csv("nstar_orders.csv")

#Used to inspect the first 5 rows to check the data has loaded correctly/overview
orders_df.head()

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


In [ ]:
#Used to check the column data types of the DataFrame
orders_df.info()

#Used to check if there are any duplicate records
"Duplicates?", orders_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               1250 non-null   object 
 1   customer_id            1250 non-null   object 
 2   service_type           1250 non-null   object 
 3   order_created_at       1250 non-null   object 
 4   promised_window_hours  1250 non-null   int64  
 5   pickup_zone            1250 non-null   object 
 6   dropoff_zone           1250 non-null   object 
 7   priority_level         1250 non-null   object 
 8   order_value            1250 non-null   float64
 9   booking_channel        1225 non-null   object 
 10  special_handling_flag  1250 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 107.6+ KB


('Duplicates?', np.int64(0))

In [ ]:
orders_before = orders_df.isnull().sum().to_frame(name="Missing Values")

format_table(orders_before, "Orders Overview")


,Missing Values
Order Id,0
Customer Id,0
Service Type,0
Order Created At,0
Promised Window Hours,0
Pickup Zone,0
Dropoff Zone,0
Priority Level,0
Order Value,0
Booking Channel,25


In [ ]:
#As per best practise, used .fillna() to replace the missing values (NaN) with "Unknown"
orders_df["booking_channel"] = orders_df["booking_channel"].fillna("Unknown")

#Reran the null count
orders_df.isnull().sum()

,0
order_id,0
customer_id,0
service_type,0
order_created_at,0
promised_window_hours,0
pickup_zone,0
dropoff_zone,0
priority_level,0
order_value,0
booking_channel,0


In [ ]:
orders_df["pickup_zone"].value_counts()

,count
pickup_zone,
East,104
South,103
EAST,103
RiverSide,86
Airport,85
WEST,84
Ctr,80
CENTRAL,79
Central,79


In [ ]:
orders_df["dropoff_zone"].value_counts()

,count
dropoff_zone,
WEST,99
South,98
RiverSide,98
West,98
AIRPORT,88
Riverside,83
EAST,80
SOUTH,80
East,75


In [ ]:
orders_df['dropoff_zone'] = zone_cleaning(orders_df['dropoff_zone'])

In [ ]:
orders_df['pickup_zone'] = zone_cleaning(orders_df['pickup_zone'])

In [ ]:
orders_df["pickup_zone"].value_counts()

,count
pickup_zone,
Central,238
East,207
South,181
North,174
West,155
Riverside,151
Airport,144


In [ ]:
orders_df["dropoff_zone"].value_counts()

,count
dropoff_zone,
West,197
North,191
Central,185
Riverside,181
South,178
Airport,163
East,155


In [ ]:
orders_df["order_created_at"] = pd.to_datetime(orders_df["order_created_at"])

2.3. Incidents dataset

In [ ]:
incidents_df = pd.read_csv("nstar_incidents.csv")

incidents_df.head()

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0
3,I0004,DL00417,ProofMissing,2025-02-09 00:16:00,Medium,Closed,9.8
4,I0005,DL00897,RouteDeviation,2025-01-04 02:49:00,Low,Open,13.0


In [ ]:
incidents_df.info()

"Duplicates?", incidents_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   incident_id        280 non-null    object 
 1   delivery_id        280 non-null    object 
 2   incident_type      280 non-null    object 
 3   reported_at        280 non-null    object 
 4   severity           280 non-null    object 
 5   resolution_status  280 non-null    object 
 6   resolved_hours     263 non-null    float64
dtypes: float64(1), object(6)
memory usage: 15.4+ KB


('Duplicates?', np.int64(0))

In [ ]:
incidents_overview = incidents_df.isnull().sum().to_frame(name="Missing Values")

format_table(incidents_overview, "Incident Overview")

,Missing Values
Incident Id,0
Delivery Id,0
Incident Type,0
Reported At,0
Severity,0
Resolution Status,0
Resolved Hours,17


In [ ]:
incidents_df["resolved_hours"].describe()

,resolved_hours
count,263.000000
mean,12.011407
std,7.751258
min,0.000000
25%,5.150000
50%,11.500000
75%,17.950000
max,41.700000


In [ ]:
incidents_df["resolved_hours_missing"] = incidents_df["resolved_hours"].notnull()

In [ ]:
resolved_missing = incidents_df["resolved_hours_missing"].value_counts().to_frame("Count")

resolved_missing["%"] = (
    resolved_missing["Count"] / resolved_missing["Count"].sum() * 100
)

resolved_missing.index = ["Avaliable", "Missing"]

format_table(resolved_missing, "Overview of the Resolved Hours")

,Count,%
Avaliable,263,93.93
Missing,17,6.07


2.4. App Events dataset

In [ ]:
app_events_df = pd.read_csv("nstar_app_events.csv")

app_events_df.head()

,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,north,301,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,SOUTH,60,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118,1
3,AE00004,C0407,O00756,2025-08-23 17:38:00,eta_refresh,S41236,iOS,CENTRAL,442,1
4,AE00005,C0506,NaN,2024-05-29 10:33:00,search_route,S12030,iOS,north,60,1


In [ ]:
app_events_df.info()

"Duplicates?", app_events_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_id         640 non-null    object
 1   customer_id      640 non-null    object
 2   order_id         496 non-null    object
 3   event_timestamp  640 non-null    object
 4   event_type       640 non-null    object
 5   session_id       640 non-null    object
 6   device_type      640 non-null    object
 7   zone_context     640 non-null    object
 8   api_latency_ms   640 non-null    int64 
 9   success_flag     640 non-null    int64 
dtypes: int64(2), object(8)
memory usage: 50.1+ KB


('Duplicates?', np.int64(0))

In [ ]:
missing_app_events = app_events_df.isnull().sum().to_frame(name="Missing Values")

format_table(missing_app_events, "App Event Overview")

,Missing Values
Event Id,0
Customer Id,0
Order Id,144
Event Timestamp,0
Event Type,0
Session Id,0
Device Type,0
Zone Context,0
Api Latency Ms,0
Success Flag,0


In [ ]:
app_events_df["order_recorded_flag"] = app_events_df["order_id"].notnull()

In [ ]:
order_flag = app_events_df["order_recorded_flag"].value_counts().to_frame("Count")

order_flag["Total %"] = (
    order_flag["Count"] / order_flag["Count"].sum() * 100
)

order_flag.index = ["Recorded", "Not recorded"]

format_table(order_flag, "Order Recorded")

,Count,Total %
Recorded,496,77.50
Not Recorded,144,22.50


In [ ]:
app_events_df["zone_context"] = zone_cleaning(app_events_df["zone_context"])

In [ ]:
app_events_df["event_timestamp"] = pd.to_datetime(app_events_df["event_timestamp"])

2.5. Deliveries dataset

In [ ]:
deliveries_df = pd.read_csv("nstar_deliveries.csv")

deliveries_df.head()

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,18/06/2024 10:57,05:59.9,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,11/01/2025 18:45,39:00.0,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,02/06/2025 20:39,45:32.4,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,08/03/2024 23:31,30:08.1,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,21/09/2025 11:43,45:34.1,OnTime,14.52,1,0,4.18,9.22


In [ ]:
deliveries_df.info()

"Duplicates?", deliveries_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   delivery_id                    950 non-null    object 
 1   order_id                       950 non-null    object 
 2   driver_id                      950 non-null    object 
 3   vehicle_id                     950 non-null    object 
 4   hub_id                         950 non-null    object 
 5   dispatch_time                  950 non-null    object 
 6   delivery_completed_at          931 non-null    object 
 7   delivery_status                950 non-null    object 
 8   route_distance_km              950 non-null    float64
 9   manual_route_override_count    950 non-null    int64  
 10  proof_of_completion_missing    950 non-null    int64  
 11  customer_rating_post_delivery  936 non-null    float64
 12  fuel_or_charge_cost            950 non-null    flo

('Duplicates?', np.int64(0))

In [ ]:
deliveries_df["route_distance_km"] = deliveries_df["route_distance_km"].round(2)

In [ ]:
deliveries_overview = deliveries_df.isnull().sum().to_frame(name="Missing Values")
format_table(deliveries_overview, "Deliveries Overview")

,Missing Values
Delivery Id,0
Order Id,0
Driver Id,0
Vehicle Id,0
Hub Id,0
Dispatch Time,0
Delivery Completed At,19
Delivery Status,0
Route Distance Km,0
Manual Route Override Count,0


In [ ]:
deliveries_df["delivery_completed_at"].describe()

,delivery_completed_at
count,931
unique,910
top,46:22.8
freq,3


In [ ]:
deliveries_df["customer_rating_post_delivery"].describe()

,customer_rating_post_delivery
count,936.000000
mean,3.864679
std,0.894420
min,1.000000
25%,3.360000
50%,4.040000
75%,4.550000
max,5.000000


In [ ]:
#Created binary flag variables to allow for True/False analysis
deliveries_df["delivery_complete_flag"] = deliveries_df["delivery_completed_at"].notnull()

deliveries_df["rating_recorded_flag"] = deliveries_df["customer_rating_post_delivery"].notnull()

2.6. Complaints dataset

In [ ]:
complaints_df = pd.read_csv("nstar_complaints.csv")

complaints_df.head()

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18


In [ ]:
complaints_df.info()

"Duplicates?", complaints_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   complaint_id         320 non-null    object 
 1   customer_id          320 non-null    object 
 2   order_id             320 non-null    object 
 3   complaint_type       320 non-null    object 
 4   channel              320 non-null    object 
 5   severity             320 non-null    object 
 6   created_at           320 non-null    object 
 7   status               320 non-null    object 
 8   resolution_days      320 non-null    int64  
 9   compensation_amount  304 non-null    float64
dtypes: float64(1), int64(1), object(8)
memory usage: 25.1+ KB


('Duplicates?', np.int64(0))

In [ ]:
complaints_overview = complaints_df.isnull().sum().to_frame(name="Missing Values")

format_table(complaints_overview, "Complaints Overview")

,Missing Values
Complaint Id,0
Customer Id,0
Order Id,0
Complaint Type,0
Channel,0
Severity,0
Created At,0
Status,0
Resolution Days,0
Compensation Amount,16


In [ ]:
resolution_overview = complaints_df["resolution_days"] \
    .agg(["min", "max"]) \
    .to_frame(name="Resolution time (days)")

resolution_overview.index = ["Min.", "Max."]

format_table(resolution_overview, "Overview: Complaint Resolution")

,Resolution Time (Days)
Min.,1
Max.,25


In [ ]:
complaints_df["complaint_type"].value_counts()

,count
complaint_type,
Delay,101
MissedPickup,64
AppIssue,53
DriverBehaviour,51
SupportExperience,20
Billing,16
Damage,15


In [ ]:
complaints_df["compensation_recorded_flag"] = complaints_df["compensation_amount"].notnull()

2.7. Drivers dataset

In [ ]:
drivers_df = pd.read_csv("nstar_drivers.csv")

drivers_df.head()

,driver_id,base_zone,employment_type,years_experience,training_score,driver_rating,shift_preference,active_flag
0,D001,AIRPORT,FullTime,8,67.8,3.54,Morning,1
1,D002,Central,FullTime,4,42.4,3.94,Evening,1
2,D003,AIRPORT,FullTime,11,96.5,5.00,Evening,1
3,D004,Airport,PartTime,13,88.9,4.75,Morning,1
4,D005,north,FullTime,3,69.7,4.14,Morning,1


In [ ]:
drivers_df.info()

"Duplicates?", drivers_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   driver_id         170 non-null    object 
 1   base_zone         170 non-null    object 
 2   employment_type   170 non-null    object 
 3   years_experience  170 non-null    int64  
 4   training_score    163 non-null    float64
 5   driver_rating     170 non-null    float64
 6   shift_preference  170 non-null    object 
 7   active_flag       170 non-null    int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 10.8+ KB


('Duplicates?', np.int64(0))

In [ ]:
drivers_overview = drivers_df.isnull().sum().to_frame(name="Missing N.")

format_table(drivers_overview, "Overview of Drivers")

,Missing N.
Driver Id,0
Base Zone,0
Employment Type,0
Years Experience,0
Training Score,7
Driver Rating,0
Shift Preference,0
Active Flag,0


In [ ]:
drivers_df["base_zone"].value_counts()

,count
base_zone,
South,21
East,14
North,13
Central,12
NORTH,12
north,11
West,10
AIRPORT,10
RiverSide,10


In [ ]:
drivers_df["base_zone"] = zone_cleaning(drivers_df["base_zone"])

In [ ]:
drivers_df["base_zone"].value_counts()

,count
base_zone,
North,36
South,29
Central,28
East,21
West,20
Airport,19
Riverside,17


In [ ]:
drivers_df["training_score"].describe()

,training_score
count,163.000000
mean,74.914724
std,11.213827
min,40.600000
25%,68.550000
50%,75.200000
75%,82.750000
max,99.000000


In [ ]:
drivers_df["missing_training_flag"] = drivers_df["training_score"].isnull()

In [ ]:
training_missing_overview = drivers_df["missing_training_flag"].value_counts().to_frame("Count")

training_missing_overview["%"] = (
    training_missing_overview["Count"] / training_missing_overview["Count"].sum() * 100
)

training_missing_overview.index = ["Avaliable", "Missing"]

format_table(training_missing_overview, "Training Score Data")

,Count,%
Avaliable,163,95.88
Missing,7,4.12


2.7. Vehicles dataset

In [ ]:
vehicles_df = pd.read_csv("nstar_vehicles.csv")

vehicles_df.head()

,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928,Active,v2.2
1,V002,EV,AIRPORT,2024-04-21 16:14:00,67.9,159368,InRepair,v2.2
2,V003,CargoVan,north,2025-11-24 23:59:00,91.7,219359,Active,v2.1
3,V004,Hybrid,RiverSide,2024-06-07 13:21:00,NaN,36310,Active,v2.2
4,V005,CargoVan,West,2025-11-15 11:08:00,58.6,146638,Active,v2.2


In [ ]:
vehicles_df.info()

"Duplicates?", vehicles_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   vehicle_id          120 non-null    object 
 1   vehicle_type        120 non-null    object 
 2   assigned_zone       120 non-null    object 
 3   commission_date     120 non-null    object 
 4   battery_health_pct  116 non-null    float64
 5   odometer_km         120 non-null    int64  
 6   maintenance_status  120 non-null    object 
 7   telematics_version  120 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 7.6+ KB


('Duplicates?', np.int64(0))

In [ ]:
vehicles_df["battery_health_pct"].describe()

,battery_health_pct
count,116.000000
mean,76.785345
std,12.698985
min,42.000000
25%,68.200000
50%,78.050000
75%,85.775000
max,100.000000


In [ ]:
vehicles_df["assigned_zone"].value_counts()

,count
assigned_zone,
AIRPORT,13
East,12
Riverside,10
CENTRAL,10
North,9
WEST,8
South,8
EAST,8
NORTH,7


In [ ]:
vehicles_df["assigned_zone"] = zone_cleaning(vehicles_df["assigned_zone"])

In [ ]:
vehicles_df["assigned_zone"].value_counts()

,count
assigned_zone,
Central,22
North,21
East,20
Riverside,16
Airport,14
South,14
West,13


In [ ]:
vehicles_overviews = vehicles_df.isnull().sum().to_frame(name="Missing Values")
format_table(vehicles_overviews, "Vehicles Overview")

,Missing Values
Vehicle Id,0
Vehicle Type,0
Assigned Zone,0
Commission Date,0
Battery Health Pct,4
Odometer Km,0
Maintenance Status,0
Telematics Version,0


In [ ]:
vehicles_df["battery_health_pct"].describe()

,battery_health_pct
count,116.000000
mean,76.785345
std,12.698985
min,42.000000
25%,68.200000
50%,78.050000
75%,85.775000
max,100.000000


In [ ]:
vehicles_df["battery_health_missing"] = vehicles_df["battery_health_pct"].isnull()

2.8. Hubs dataset

In [ ]:
hubs_df = pd.read_csv("nstar_hubs.csv")

hubs_df.head()

,hub_id,hub_name,zone,hub_type,capacity_score
0,H01,North Exchange,North,Dispatch,82
1,H02,South Link,South,Dispatch,78
2,H03,East Dock,East,Warehouse,74
3,H04,West Gate,West,Dispatch,69
4,H05,Central Core,Central,Control,88


In [ ]:
hubs_df.info()

"Duplicates?", hubs_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   hub_id          8 non-null      object
 1   hub_name        8 non-null      object
 2   zone            8 non-null      object
 3   hub_type        8 non-null      object
 4   capacity_score  8 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 452.0+ bytes


('Duplicates?', np.int64(0))

In [ ]:
hubs_df["zone"] = zone_cleaning(hubs_df["zone"])

In [ ]:
hubs_overview = hubs_df.isnull().sum().to_frame(name="Missing Values")
format_table(hubs_overview, "Hubs Overview")

,Missing Values
Hub Id,0
Hub Name,0
Zone,0
Hub Type,0
Capacity Score,0


2.9. Data Dictionary


In [ ]:
data_dict_df = pd.read_csv("nstar_data_dictionary.csv")

data_dict_df.head()

,file_name,record_count,description
0,hubs.csv,8,Operational hubs and control points
1,customers.csv,650,Customer master data with engagement and loyal...
2,drivers.csv,170,Driver workforce data with training and rating...
3,vehicles.csv,120,Fleet asset data including battery and mainten...
4,orders.csv,1250,Service orders across mobility and logistics o...


2.10 Dataset Overview

In [ ]:
complaints_deliveries_df = complaints_df.merge(
    deliveries_df,
    on="order_id",
    how="left",
    indicator=True
)

format_table(
    complaints_deliveries_df["_merge"].value_counts().to_frame("Count"),
    "Complaint-to-Delivery Record Linkage"
)

,Count
Both,232
Left Only,88
Right Only,0


In [ ]:
#Exporting the cleaned datasets for use in the other analytical sections

orders_df.to_csv("vehicles_clean.csv", index=False)

deliveries_df.to_csv("deliveries_clean.csv", index=False)

vehicles_df.to_csv("vehicles_clean.csv", index=False)

hubs_df.to_csv("hubs_clean.csv", index=False)

customers_df.to_csv("customers_clean.csv", index=False)

incidents_df.to_csv("incidents_clean.csv", index=False)

app_events_df.to_csv("app_events_clean.csv", index=False)

complaints_df.to_csv("complaints_clean.csv", index=False)

drivers_df.to_csv("drivers_clean.csv", index=False)




